## Step 1: Import libraries

In [ ]:
import ast
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

credits = credits.rename(columns={"movie_id": "id"}).drop(columns="title")
df = movies.merge(credits, on="id")

# vote_average / vote_count are used later for quality filtering
df = df[["id", "title", "overview", "genres", "keywords", "cast", "crew",
         "vote_average", "vote_count"]]
print("Shape:", df.shape)
df.head(3)

In [ ]:
print("Missing values:\n", df.isnull().sum(), "\n")

df["overview"] = df["overview"].fillna("")
df = df.drop_duplicates(subset="title").reset_index(drop=True)
print("After cleaning:", df.shape)

In [ ]:
def get_names(text, top_n=None):
    try:
        names = [item["name"] for item in ast.literal_eval(text)]
    except (ValueError, SyntaxError):
        return []
    return names[:top_n] if top_n else names

def get_director(text):
    try:
        return [p["name"] for p in ast.literal_eval(text) if p["job"] == "Director"]
    except (ValueError, SyntaxError):
        return []

def squash(names):
    return [n.replace(" ", "").lower() for n in names]

df["genres"]   = df["genres"].apply(get_names).apply(squash)
df["keywords"] = df["keywords"].apply(get_names).apply(squash)
df["cast"]     = df["cast"].apply(lambda x: get_names(x, top_n=3)).apply(squash)
df["director"] = df["crew"].apply(get_director).apply(squash)

df["metadata"] = df.apply(
    lambda r: " ".join(r["genres"] + r["keywords"] + r["cast"] + r["director"]), axis=1
)
df[["title", "metadata"]].head(3)

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

sample = df["overview"][0]
print("BEFORE:", sample[:200])
print("AFTER :", preprocess_text(sample)[:200])

In [ ]:
df["overview_clean"] = df["overview"].apply(preprocess_text)

# Feature weighting: genres, keywords & director count double, cast counts once
join = lambda col: df[col].apply(" ".join)
df["tags"] = (
    df["overview_clean"] + " "
    + join("genres") + " " + join("genres") + " "
    + join("keywords") + " " + join("keywords") + " "
    + join("director") + " " + join("director") + " "
    + join("cast")
)
df[["title", "tags"]].head(3)

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["tags"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.60)

df["weighted_rating"] = (
    df["vote_count"] / (df["vote_count"] + m) * df["vote_average"]
    + m / (df["vote_count"] + m) * C
)
print(f"Mean rating C = {C:.2f}, vote threshold m = {m:.0f}")

In [ ]:
title_to_idx = pd.Series(df.index, index=df["title"].str.lower())
MIN_VOTES = 100

def find_index(title):
    key = title.lower().strip()
    if key not in title_to_idx.index:
        match = difflib.get_close_matches(key, title_to_idx.index, n=1, cutoff=0.6)
        if not match:
            print(f"'{title}' not found - skipped.")
            return None
        print(f"'{title}' not found. Using '{match[0].title()}' instead.")
        key = match[0]
    return int(title_to_idx[key])

def recommend(titles, n=10, quality=True):
    if isinstance(titles, str):
        titles = [titles]
    idxs = [i for i in (find_index(t) for t in titles) if i is not None]
    if not idxs:
        return "No matching movies found in the dataset."

    # average the vectors of all liked movies, then compare with every movie
    user_vec = np.asarray(tfidf_matrix[idxs].mean(axis=0))
    scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
    scores[idxs] = -1                                  # never recommend the input movies

    top = scores.argsort()[::-1][: 50 if quality else n]
    res = pd.DataFrame({
        "Movie": df["title"].iloc[top].values,
        "Similarity": scores[top],
        "Rating": df["vote_average"].iloc[top].values,
        "Votes": df["vote_count"].iloc[top].values,
        "wr": df["weighted_rating"].iloc[top].values,
    })

    if quality:
        res = res[res["Votes"] >= MIN_VOTES].copy()
        res["final"] = 0.7 * (res["Similarity"] / res["Similarity"].max()) + 0.3 * (res["wr"] / 10)
        res = res.sort_values("final", ascending=False)

    res = res.head(n).drop(columns=["wr", "final"], errors="ignore")
    res["Similarity"] = res["Similarity"].round(3)
    return res.reset_index(drop=True)

In [ ]:
recommend("Avatar")

In [ ]:
# same movie, but WITHOUT quality filtering - compare with the output above
recommend("Avatar", quality=False)

In [ ]:
# multi-movie input: recommendations for someone who likes all three
recommend(["Avatar", "Interstellar", "Inception"], n=8)

In [ ]:
recommend("The Dark Knight Rises", n=5)

In [ ]:
recommend("Toy Story", n=5)

In [ ]:
genre_sets = df["genres"].apply(set)
eligible = df[df["genres"].str.len() > 0].index
sample_idx = np.random.RandomState(42).choice(eligible, size=min(300, len(eligible)), replace=False)

def precision_at_k(matrix, k=10):
    sims = cosine_similarity(matrix[sample_idx], matrix)
    scores = []
    for row, i in zip(sims, sample_idx):
        row[i] = -1
        top = np.argsort(row)[::-1][:k]
        hits = sum(1 for j in top if genre_sets[i] & genre_sets[j])
        scores.append(hits / k)
    return float(np.mean(scores))

vec_kwargs = dict(max_features=10000, ngram_range=(1, 2))
configs = {
    "Overview only (Count)":   CountVectorizer(**vec_kwargs).fit_transform(df["overview_clean"]),
    "Overview only (TF-IDF)":  TfidfVectorizer(**vec_kwargs).fit_transform(df["overview_clean"]),
    "Metadata only (TF-IDF)":  TfidfVectorizer(**vec_kwargs).fit_transform(df["metadata"]),
    "Combined, weighted (TF-IDF)": tfidf_matrix,
}

results = pd.Series({name: precision_at_k(mat) for name, mat in configs.items()}, name="Precision@10")
print(results.round(3))

results.plot(kind="barh", figsize=(8, 3.5), title="Genre-based precision@10 by configuration")
plt.xlabel("Precision@10")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
!pip -q install gradio
import gradio as gr

def recommend_ui(titles, n, quality):
    names = [t.strip() for t in titles.split(",") if t.strip()]
    res = recommend(names, int(n), quality)
    return res if isinstance(res, pd.DataFrame) else pd.DataFrame({"Message": [res]})

demo = gr.Interface(
    fn=recommend_ui,
    inputs=[
        gr.Textbox(label="Movie(s) you like (comma-separated)", value="Avatar"),
        gr.Slider(3, 20, value=10, step=1, label="Number of recommendations"),
        gr.Checkbox(value=True, label="Prefer well-rated movies"),
    ],
    outputs=gr.Dataframe(label="Recommendations"),
    title="🎬 Movie Recommender",
    description="Content-based recommendations using TF-IDF + cosine similarity.",
)
demo.launch()